# Conference page mock-up — all figures on one page

Places the figures on an A4 page with IEEE two-column margins (text area 181.4 × 235 mm), a draft caption under each, and reports whether the stack fits the text-area height. Row 1 holds two figures side by side, each with its own caption; Figs. 3–4 span the full 178 mm.

- **Fig. 1** (row 1, left, 58 mm) experiment — `figures/out/conf_fig2/conf_fig2.png` (`conf_fig2_experiment.ipynb`; placeholders until `figures/exp/{a..e}.png` exist).
- **Fig. 2** (row 1, right, 116 mm) concept — `figures/out/conf_fig1/conf_fig1.png` (`conf_fig1_concept.ipynb`: author's PowerPoint export `figures/out/fig1.png`).
- **Fig. 3** DiffCSP + projected energy — `figures/out/conf_fig3/conf_fig3.png`.
- **Fig. 4** ChargE3Net — `figures/out/conf_fig4/conf_fig4.png`.

Figure numbers follow the reading order (left before right). Re-run the figure notebooks first; this notebook only reads their PNGs.

In [ ]:
import sys, textwrap
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from style import *
OUT = FIG_OUT / 'conf_page'
FW = W_CONF / MM                 # 178 mm
GAP_ROW1 = 4.0                   # mm between the two figures of row 1
CAPTIONS = {
    'exp': "Fig. 1. Fabricated PCM device. (a) Device structure. (b) Cross-sectional TEM. (c) Material "
           "characterization. (d) Electrical switching. (e) Multilevel programming / retention.",
    'concept': "Fig. 2. Overview: PCM conductance states set the weight levels of a crossbar on which two pretrained "
               "materials models, diffusion crystal-structure generation and GNN charge-density prediction, reuse the "
               "same programmed weights over thousands of evaluations.",
    'fig3': "Fig. 3. Crystal-structure generation (DiffCSP) on simulated PCM arrays. (a) Projected energy vs a GPU. "
            "(b) Noise-budget sampling keeps the digital accuracy as read noise grows. (c) Read noise is absorbed into "
            "the diffusion noise during denoising. (d) Analog-generated GeTe–Sb₂Te₃ structures reach the digital mixing "
            "energies (last 100 steps digital).",
    'fig4': "Fig. 4. Charge-density prediction (ChargE3Net) on simulated PCM arrays (analog: average of three arrays). "
            "(a–c) The arrays add a small, structure-independent density error. (d) The predicted densities reproduce "
            "the DFT density of states.",
}
# rows of (key, png, x offset in mm from the figure-block left edge, placed width in mm)
EXP_W = 58.0; CON_W = FW - EXP_W - GAP_ROW1
ROWS = [[('exp', FIG_OUT / 'conf_fig2' / 'conf_fig2.png', 0.0, EXP_W),
         ('concept', FIG_OUT / 'conf_fig1' / 'conf_fig1.png', EXP_W + GAP_ROW1, CON_W)],
        [('fig3', FIG_OUT / 'conf_fig3' / 'conf_fig3.png', 0.0, FW)],
        [('fig4', FIG_OUT / 'conf_fig4' / 'conf_fig4.png', 0.0, FW)]]


In [ ]:
# page geometry (mm): A4 with IEEE conference margins -> text area 181.4 x 235 mm
PAGE_W, PAGE_H = 210.0, 297.0
M_TOP, M_BOTTOM, M_SIDE = 19.0, 43.0, 14.3
TEXT_W, TEXT_H = PAGE_W - 2 * M_SIDE, PAGE_H - M_TOP - M_BOTTOM
FIG_W = FW                               # 178 mm figure block, centred in the text width
CAP_PT, CAP_LEAD = 8, 1.15              # IEEE caption: 8 pt, line height 1.15 x
GAP_FIG_CAP, GAP_CAP_FIG = 1.0, 2.5     # mm
CAP_FONT = ['Times New Roman', 'Times', 'DejaVu Serif']

def wrap_lines(fig, text, width_mm):
    """Greedy word wrap measured with the renderer at the caption font."""
    r = fig.canvas.get_renderer()
    probe = fig.text(0, 0, '', fontsize=CAP_PT, family=CAP_FONT)
    lines, cur = [], ''
    for w in text.split():
        trial = (cur + ' ' + w).strip()
        probe.set_text(trial)
        if probe.get_window_extent(r).width / fig.dpi * 25.4 > width_mm and cur:
            lines.append(cur); cur = w
        else:
            cur = trial
    probe.remove()
    return lines + [cur]

In [ ]:
# ---------------- 拼版配色 ----------------
Cp_TEXTAREA = '#9E9E9E'   # 版心虚线框
Cp_OVER = '#D62728'       # 超出版心时的红线
# ------------------------------------------
fig = plt.figure(figsize=(PAGE_W * MM, PAGE_H * MM), dpi=150)
fig.canvas.draw()
mm_x = lambda x: x / PAGE_W
mm_y = lambda y: 1 - y / PAGE_H          # y measured from the page top
x0 = M_SIDE + (TEXT_W - FIG_W) / 2
y = M_TOP
rows = []
line_mm = CAP_PT * CAP_LEAD * 25.4 / 72
for row in ROWS:
    y_img, row_cap = y, 0.0
    hs = []
    for key, path, dx, wmm in row:
        img = plt.imread(path)
        h = wmm * img.shape[0] / img.shape[1]; hs.append(h)
        ax = fig.add_axes([mm_x(x0 + dx), mm_y(y_img + h), wmm / PAGE_W, h / PAGE_H]); ax.imshow(img); ax.axis('off')
    h_row = max(hs)
    for (key, path, dx, wmm), h in zip(row, hs):
        lines = wrap_lines(fig, CAPTIONS[key], wmm)
        yc = y_img + h_row + GAP_FIG_CAP
        for k, ln in enumerate(lines):
            fig.text(mm_x(x0 + dx), mm_y(yc + (k + 0.8) * line_mm), ln, fontsize=CAP_PT, family=CAP_FONT, ha='left',
                     va='baseline')
        row_cap = max(row_cap, len(lines) * line_mm)
        rows.append((key, round(wmm, 1), round(h, 1), len(lines), round(len(lines) * line_mm, 1)))
    y = y_img + h_row + GAP_FIG_CAP + row_cap + GAP_CAP_FIG
used = y - GAP_CAP_FIG - M_TOP
frame = plt.Rectangle((mm_x(M_SIDE), mm_y(M_TOP + TEXT_H)), TEXT_W / PAGE_W, TEXT_H / PAGE_H, transform=fig.transFigure,
                      fill=False, ls=(0, (4, 3)), lw=0.5, ec=Cp_TEXTAREA)
fig.add_artist(frame)
if used > TEXT_H:
    fig.add_artist(plt.Line2D([0, 1], [mm_y(M_TOP + TEXT_H)] * 2, color=Cp_OVER, lw=0.8))
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / 'conf_page.png', dpi=300)
plt.close(fig)
budget = pd.DataFrame(rows, columns=['figure', 'width_mm', 'height_mm', 'caption_lines', 'caption_mm'])
print(budget.to_string(index=False))
print(f'used {used:.1f} mm of {TEXT_H:.1f} mm text height -> {"FITS" if used <= TEXT_H else f"OVER by {used - TEXT_H:.1f} mm"}')
display(NotebookImage(filename=str(OUT / 'conf_page.png'), width=620))